Question 1: 

ist fachgerechte sondermüll entsorgung in zürich zu teuer, müsste die stadt zürich einen billigen entsorgungsdienst anbieten? korrelieren die anzahl illegal entsorgten abfälle mit dem steuerbaren Einkommen der Bevölkerung

In [1]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import requests
import datetime as dt

In [2]:
#Access and load the züri wie neu data via API

url_zwn = "https://www.ogd.stadt-zuerich.ch/wfs/geoportal/Zueri_wie_neu?service=WFS&version=1.1.0&request=GetFeature&outputFormat=GeoJSON&typename=zwn_meldungen_p"
response_zwn = requests.get(url_zwn)
if response_zwn.status_code == 200:
    print("Data züri wie neu loaded sugessfull")
    zurich_zwn_gdf = gpd.read_file(response_zwn.url)
else:
    print("Data züri wie neu loading failed")

Data züri wie neu loaded sugessfull


In [3]:
url_neighbourhoods = "https://www.ogd.stadt-zuerich.ch/wfs/geoportal/Statistische_Quartiere?service=WFS&version=1.1.0&request=GetFeature&outputFormat=GeoJSON&typename=adm_statistische_quartiere_map"
response_neighbourhoods = requests.get(url_neighbourhoods)
if response_neighbourhoods.status_code == 200:
    print("Data neighbourhood loaded sugessfull")
    zurich_neighbourhoods_gdf = gpd.read_file(response_neighbourhoods.url)
else:
    print("Data neighbourhood loading failed")

Data neighbourhood loaded sugessfull


In [20]:

#Access and load the income data via API

url_income = 'https://data.stadt-zuerich.ch/api/3/action/datastore_search'
parameter_income = {"resource_id":"af01ed91-04f8-445b-8dfc-04cbf0a27e95", "limit":3000}
response_income = requests.get(url_income, params=parameter_income)
if response_income.status_code == 200:
    print("Data Income loaded sugessfull")

    #convert the json response into a dictionary (because the CKAN API returns a new dictionary where the data is in result and record)
    zurich_income_dic = response_income.json()["result"]["records"]
    #convert the dictionary into a dataframe
    zurich_income_df = pd.DataFrame(zurich_income_dic)
else:
    print("Data income loading failed")


Data Income loaded sugessfull


In [ ]:

## filter income for the newest year

# display the current year
current_year = int(dt.date.today().year)

#filter dataframe for the newest year (always current year minus 3)
zurich_income_df = zurich_income_df[zurich_income_df["StichtagDatJahr"] == str(current_year -3)].copy()


In [ ]:
## create a mean for the 3 steuer tarife

#clean the data (drop na, convert to float)
zurich_income_df.dropna(subset=["SteuerEinkommen_p50"])
zurich_income_df["SteuerEinkommen_p50"] = zurich_income_df["SteuerEinkommen_p50"].astype(float)

#group by neigbourhoods and compute the mean income (of the 3 steuertarife) for each income
grouped_by_neighbourhoods= zurich_income_df.groupby("QuarLang")
zurich_mean_income_df = grouped_by_neighbourhoods[["SteuerEinkommen_p50"]].mean()



In [56]:
# Sort the züri wie neu data for waste

print(zurich_zwn_gdf["service_name"].unique())

zurich_zwn_waste_gdf = zurich_zwn_gdf[zurich_zwn_gdf["service_name"]=="Abfall/Sammelstelle"]

print(zurich_zwn_waste_gdf["service_name"].unique())



<ArrowStringArray>
[   'Strasse/Trottoir/Platz',       'Abfall/Sammelstelle',
   'Grünflächen/Spielplätze',         'Beleuchtung/Uhren',
                  'Graffiti', 'Signalisation/Lichtsignal',
         'Brunnen/Hydranten',                    'VBZ/ÖV',
                 'Allgemein',                'Schädlinge']
Length: 10, dtype: str
<ArrowStringArray>
['Abfall/Sammelstelle']
Length: 1, dtype: str
